# HW2: CIFAR-100 Image Classification (ConvNeXt-Tiny + DINOv2 ViT-g ensemble)

샘플 LeNet5 코드(public 약 0.228)에서 시작해서, 모델 두 개를 한 session 안에서 학습하고 예측을 합쳤음.

1. ConvNeXt-Tiny fine-tuning (2~8번): ImageNet pretrained CNN 전체를 CIFAR-100으로 10 epoch 학습
2. DINOv2 ViT-g + MLP head (9~11번): ViT-g는 학습 안 하고(frozen) feature만 뽑고, 그 위에 작은 MLP head만 학습
3. ensemble (12번): 두 모델 softmax 확률 평균 내서 submission.csv 만듦

실행 방법
- Colab 무료, 런타임 T4 GPU
- 왼쪽 파일 탭에 2026_02_AI_train, 2026_02_AI_test 올리고 위에서부터 순서대로 실행
- 전체 2시간 정도 걸림 (ConvNeXt 학습 30분, ViT-g feature 추출 70분 정도)
- 학습/예측은 전부 이 session 안에서 함. 전에 저장해둔 weight, feature, logits 안 씀

pretrained weight
- ConvNeXt: timm `convnext_tiny.fb_in22k_ft_in1k` (ImageNet)
- DINOv2: timm `vit_giant_patch14_dinov2.lvd142m` (LVD-142M)
- CIFAR-100으로 학습된 weight는 안 씀

## 0. 데이터셋 클래스 (샘플 코드, 수정 X)

pickle에서 이미지(N x 3072) 읽어서 N x 32 x 32 x 3으로 바꿔줌. test는 label 없이 이미지만 나옴.

In [1]:
#####################
### DO NOT MODIFY ###
#####################
import os
import pickle
import numpy as np
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(2302)

train_file = '2026_02_AI_train'
test_file = "2026_02_AI_test"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class cifar100_dataset(Dataset):
    def __init__(self, file_path, transform=None, is_test=False):
        self.is_test = is_test
        self.transform = transform

        with open(file_path, "rb") as f:
            dictionary = pickle.load(f, encoding="bytes")

        raw_data = dictionary[b"data"]
        self.images = raw_data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

        if not self.is_test:
            self.labels = dictionary[b"fine_labels"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image
        return image, self.labels[idx]

## 1. 데이터 파일, GPU 확인

- 런타임 새로 켤 때마다 train, test 파일 다시 올려야 함 (초기화되면 지워짐)
- 파일 없거나 GPU 아니면 학습 들어가기 전에 바로 멈추게 해둠

In [ ]:
# 파일 업로드 안 했으면 여기서 멈춤
for file_name in [train_file, test_file]:
    assert os.path.exists(file_name), f"{file_name} 파일이 없음 → 왼쪽 파일 탭에서 업로드"

# 처음에 CPU로 돌려서 시간 날린 적 있어서 넣음
assert torch.cuda.is_available(), "GPU가 없음 → 런타임 > 런타임 유형 변경 > T4 GPU"
print("데이터 파일:", [train_file, test_file])
print("GPU:", torch.cuda.get_device_name(0))

데이터 파일: ['2026_02_AI_train', '2026_02_AI_test']
GPU: Tesla T4


## 2. Transform (augmentation, 정규화)

augmentation은 train에만 적용. 클래스당 400장 정도밖에 없어서 그냥 학습하면 이미지를 외워버림(overfitting). 매 epoch 조금씩 바꿔서 넣어주면 다른 사진 보는 효과가 남.

- RandomCrop(32, padding=4): 4px 패딩 붙이고 랜덤 위치에서 32x32로 자름. 물체 위치가 좀 달라져도 맞추게 됨
- RandomHorizontalFlip: 50% 확률로 좌우 반전
- TrivialAugmentWide: 회전, 밝기, 대비, 기울이기 같은 것 중 하나를 랜덤 세기로 적용. 따로 정할 값이 없어서 편함

정규화는 샘플의 CIFAR-10 값 말고 ImageNet 평균/표준편차로 바꿈. pretrained 모델이 이 값으로 정규화된 이미지로 학습해서 똑같이 맞춰줘야 함.

test는 변형 없이 원본 그대로 넣음.

augmentation은 32x32 상태에서 CPU로 하고, 224 확대는 다음 셀에서 GPU로 함.

In [3]:
# ImageNet 평균/표준편차 (pretrained 모델 학습할 때 쓴 값)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose(
    [
        transforms.ToPILImage(),                  # augmentation 함수들이 PIL 이미지를 받음
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

# val, test용 (augmentation 없음)
test_transform = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

## 3. 이미지 확대 (32x32 -> 224x224, GPU)

- pretrained 모델은 ImageNet 224x224로 학습됨. 32x32를 그대로 넣으면 첫 층(4x4 conv, stride 4) 지나자마자 8x8이 돼서 남는 정보가 거의 없음
- 그래서 bilinear로 224까지 키워서 넣음. 정보가 새로 생기는 건 아니지만 모델이 익숙한 크기로 보이니까 pretrained 특징을 잘 씀
- transform(CPU)에서 키우면 Colab CPU 2개로는 너무 느려서 GPU가 놀게 됨. batch를 GPU로 옮긴 다음에 한 번에 키움
- IMG_SIZE를 192, 160으로 줄이면 빨라지는데 정확도는 조금 떨어질 수 있음

In [4]:
import torch.nn as nn
import torch.nn.functional as F

IMG_SIZE = 224  # pretrained 해상도

def upscale(images):
    # (batch, 3, 32, 32) -> (batch, 3, IMG_SIZE, IMG_SIZE)
    return F.interpolate(images, size=IMG_SIZE, mode="bilinear", align_corners=False)

## 4. train 90% / validation 10% 분할

- test는 정답이 없고 Kaggle public score는 하루 10번밖에 못 봄
- 그래서 train에서 10% 떼서 validation으로 쓰고 epoch마다 성능 확인함
- val은 augmentation 없이 봐야 해서 같은 train 파일로 dataset을 두 개 만듦 (full_train은 augmentation O, full_val은 X)
- 두 dataset에서 안 겹치는 인덱스만 Subset으로 골라 씀
- batch size는 224 해상도에서 T4(16GB)에 여유 있게 들어가는 64로 함

In [5]:
from torch.utils.data import Subset

BATCH_SIZE = 64

full_train = cifar100_dataset(train_file, transform=train_transform)  # augmentation O
full_val = cifar100_dataset(train_file, transform=test_transform)     # 같은 데이터, augmentation X
test_dataset = cifar100_dataset(test_file, transform=test_transform, is_test=True)

# 섞어서 앞 10%를 val로. seed 고정이라 분할은 매번 같음
indices = torch.randperm(len(full_train)).tolist()
n_val = len(indices) // 10
train_dataset = Subset(full_train, indices[n_val:])
val_dataset = Subset(full_val, indices[:n_val])
print(f"train: {len(train_dataset)}, validation: {len(val_dataset)}, test: {len(test_dataset)}")

# Colab CPU가 2개라 num_workers=2
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)

train: 36000, validation: 4000, test: 10000


## 5. 모델: ConvNeXt-Tiny (ImageNet pretrained)

transfer learning (fine-tuning)
- 처음부터 학습하면 train 4만 장으로 사물 보는 법부터 배워야 해서 데이터가 부족함
- 큰 데이터로 이미 학습된 모델은 선, 모양, 질감 같은 특징을 이미 앎
- 마지막 분류층(1000개)만 100개짜리로 바꾸고 전체를 작은 lr로 조금 더 학습함

pretrained weight: ImageNet-22K -> 1K
- 22K(1400만 장, 2만 2천 클래스)로 먼저 학습하고 1K(128만 장, 1000 클래스)로 다시 학습한 weight
- 1K로만 학습한 것보다 본 이미지가 훨씬 많아서 다른 데이터셋에 옮겨 쓸 때 보통 더 잘 나옴
- 구조는 똑같고 weight만 달라서 학습 시간은 같음
- timm 라이브러리로 불러옴

ConvNeXt는 ResNet을 요즘 방식으로 고친 CNN
- 뼈대는 ResNet이랑 같음. stage마다 크기 줄이고 채널 늘리고, 블록마다 skip connection(F(x) + x) 있음
- 구성 (224 입력 기준)
  - stem: 4x4 conv, stride 4 -> 96 x 56 x 56
  - stage 1~4: 블록 3, 3, 9, 3개 / 채널 96, 192, 384, 768 / 크기 56, 28, 14, 7 (stage 사이에 2x2 conv stride 2)
  - 끝: global average pooling, LayerNorm, Linear(768 -> 100)
- 블록 하나: 7x7 depthwise conv, LayerNorm, 1x1 conv(채널 4배), GELU, 1x1 conv(원래 채널), + x
  - depthwise conv: 채널마다 따로 conv함. 7x7처럼 kernel 커도 계산량이 적음
  - 1x1 conv: 위치마다 채널끼리 섞어줌 (depthwise가 못 하는 부분)
  - LayerNorm: BatchNorm이랑 비슷한데 batch 말고 이미지 한 장의 각 위치에서 채널 방향으로 평균/분산 구함. batch 크기 영향 X
  - GELU: ReLU랑 비슷한데 0 근처에서 부드럽게 꺾임
- stochastic depth(drop_path_rate=0.1) 씀. 학습 중에 블록을 확률적으로 건너뛰고(x만 통과) 평가 때는 다 씀. 블록 단위 Dropout 같은 느낌

In [6]:
!pip install -q timm
import timm

# 22K -> 1K로 학습된 ConvNeXt-Tiny
# num_classes=100으로 하면 마지막 층만 Linear(768 -> 100)으로 새로 만들어짐 (여기만 랜덤 초기화)
model = timm.create_model(
    "convnext_tiny.fb_in22k_ft_in1k", pretrained=True, num_classes=100, drop_path_rate=0.1,
)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"parameter 수: {n_params / 1e6:.1f}M")
print(model.head)

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

parameter 수: 27.9M
NormMlpClassifierHead(
  (global_pool): SelectAdaptivePool2d(pool_type=avg, flatten=Identity())
  (norm): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True, bias=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (pre_logits): Identity()
  (drop): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=768, out_features=100, bias=True)
)


## 6. 학습 설정

- Loss: CrossEntropyLoss, label smoothing 0.1
  - 정답을 1.0 말고 0.9로, 나머지 0.1은 다른 클래스에 나눠준 target으로 학습
  - train 이미지에 너무 확신하는 걸 막아서 overfitting 줄여줌
- Optimizer: AdamW (lr 1e-4, weight decay 0.05)
  - Adam인데 weight decay를 gradient랑 따로 적용하는 버전. ConvNeXt 논문도 AdamW 씀
  - pretrained weight 망가지지 않게 lr은 작게 1e-4
- OneCycleLR (샘플이랑 같은 scheduler)
  - 처음 10%(pct_start=0.1) 동안 lr을 천천히 올리고(warmup), 그 뒤엔 cosine으로 줄임
  - 새로 만든 분류층이 랜덤이라 초반 gradient가 큼. 처음부터 lr 크면 pretrained weight가 망가질 수 있어서 warmup 함
  - Adam momentum(beta1)은 건드리지 않게 cycle_momentum=False
- AMP (Automatic Mixed Precision)
  - 계산 일부를 float16으로 해서 T4에서 빠르게 돌리고 메모리 아낌
  - float16은 작은 gradient가 0이 돼버릴 수 있어서 GradScaler가 loss를 키워서 backward하고 다시 나눠서 업데이트함
- EPOCHS = 10: pretrained라 금방 수렴함. 첫 epoch 시간 보고 조정

In [7]:
import torch.optim as optim

EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# batch마다 step 부를 거라서 steps_per_epoch 넣어줌
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE, steps_per_epoch=len(train_loader), epochs=EPOCHS,
    pct_start=0.1, cycle_momentum=False,
)
scaler = torch.amp.GradScaler("cuda")  # AMP용

## 7. 학습 루프

샘플 학습 루프에서 바꾼 것
- batch를 GPU로 옮긴 다음 upscale()로 224로 키움
- forward는 torch.autocast 안에서 (AMP), backward랑 업데이트는 scaler 거쳐서
- epoch 끝날 때마다 train loss/acc, val acc, 걸린 시간 출력. val 볼 때는 model.eval()로 stochastic depth 끔
- 학습 끝나면 epoch별 기록을 표로 출력. gap = train acc - val acc, 이게 계속 커지면 overfitting
- val acc 제일 높았던 weight를 convnext_best.pt로 저장해두고 8번에서 불러서 씀 (같은 session 안에서만)

In [8]:
import time
from tqdm.auto import tqdm

BEST_PATH = "convnext_best.pt"  # 이 session 안에서만 씀
best_val_acc = 0.0
history = []  # 마지막에 표로 출력하려고 모아둠

for epoch in range(EPOCHS):
    start = time.time()

    # ---------- train ----------
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, labels in pbar:
        images = upscale(images.to(device, non_blocking=True))  # GPU에서 224로 키움
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # OneCycleLR라 batch마다

        total_loss += loss.item() * labels.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(
            {"Loss": f"{total_loss/total:.4f}", "Acc": f"{100.*correct/total:.2f}%"}
        )

    # ---------- validation ----------
    model.eval()  # stochastic depth 끔
    val_correct = 0
    val_total = 0
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
        for images, labels in val_loader:
            images = upscale(images.to(device, non_blocking=True))
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)
    val_acc = 100. * val_correct / val_total

    # best면 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_PATH)
    elapsed = (time.time() - start) / 60
    train_loss = total_loss / total
    train_acc = 100. * correct / total
    history.append((epoch + 1, train_loss, train_acc, val_acc, elapsed))
    print(
        f"Epoch {epoch+1}: train loss {train_loss:.4f}, train acc {train_acc:.2f}%, "
        f"val acc {val_acc:.2f}% (best {best_val_acc:.2f}%), {elapsed:.1f}분"
    )

# ---------- 결과 정리 ----------
# train acc는 augmentation, stochastic depth 켜진 상태라 val acc보다 낮게 나올 수 있음
print(f"\n{'epoch':>5} {'train_loss':>10} {'train_acc':>9} {'val_acc':>7} {'gap':>6} {'min':>5}")
for ep, tl, ta, va, t in history:
    print(f"{ep:>5} {tl:>10.4f} {ta:>9.2f} {va:>7.2f} {ta - va:>6.2f} {t:>5.1f}")
print(f"best val acc: {best_val_acc:.2f}%, 총 학습 시간: {sum(h[4] for h in history):.1f}분")

Epoch 1/10:   0%|          | 0/563 [00:00<?, ?it/s]

Epoch 1: train loss 3.5461, train acc 29.19%, val acc 81.72% (best 81.72%), 3.1분


Epoch 2/10:   0%|          | 0/563 [00:00<?, ?it/s]

Epoch 2: train loss 1.8716, train acc 68.69%, val acc 86.20% (best 86.20%), 3.0분


Epoch 3/10:   0%|          | 0/563 [00:00<?, ?it/s]

Epoch 3: train loss 1.5854, train acc 75.54%, val acc 88.17% (best 88.17%), 3.0분


... (15 outputs left out)

Epoch 9: train loss 1.2317, train acc 86.38%, val acc 91.00% (best 91.05%), 3.0분


Epoch 10/10:   0%|          | 0/563 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b49a933c9a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b49a933c9a0>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers

Traceback (most recent call last):
    if w.is_alive():  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__

      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()    
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
assert self._parent_pid == os.getpid(), 'can only test a child process'
    AssertionErrorif w.is_alive():: 
can only test a child process  File "/usr/lib/p

Epoch 10: train loss 1.2125, train acc 87.17%, val acc 90.90% (best 91.05%), 3.0분

epoch train_loss train_acc val_acc    gap   min
    1     3.5461     29.19   81.72 -52.53   3.1
    2     1.8716     68.69   86.20 -17.51   3.0
    3     1.5854     75.54   88.17 -12.63   3.0
    4     1.4585     79.23   89.45 -10.22   3.0
    5     1.3758     81.61   89.85  -8.24   3.0
    6     1.3218     83.23   90.20  -6.97   3.0
    7     1.2743     84.94   90.78  -5.83   3.0
    8     1.2398     86.15   91.05  -4.90   3.0
    9     1.2317     86.38   91.00  -4.62   3.0
   10     1.2125     87.17   90.90  -3.73   3.0
best val acc: 91.05%, 총 학습 시간: 29.9분


## 8. ConvNeXt 추론

- 7번에서 저장한 best weight 불러서 val, test 예측
- TTA: 원본이랑 좌우 반전 이미지 두 번 예측해서 logits 더함. 학습 때 flip을 했으니까 반전 이미지도 잘 맞추고, 두 개 합치면 좀 더 안정적임
- 결과는 cnx_val_logits, cnx_test_logits에 저장해두고 12번에서 DINOv2랑 합침
- test_loader가 shuffle=False라 test 순서 그대로 나옴

In [9]:
from tqdm.auto import tqdm

def predict(loader):
    # TTA(원본 + 좌우 반전) logits 모아서 반환
    model.eval()
    all_logits = []
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
        for batch in tqdm(loader, desc="Predicting"):
            # val은 (이미지, label), test는 이미지만 나와서 나눠줌
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = upscale(images.to(device, non_blocking=True))

            logits = model(images) + model(torch.flip(images, dims=[3]))  # dims=[3]이 가로 방향
            all_logits.append(logits.float().cpu())
    return torch.cat(all_logits)  # (이미지 수, 100)

model.load_state_dict(torch.load(BEST_PATH))
val_logits = predict(val_loader)
test_logits = predict(test_loader)

# val_loader가 shuffle=False라 indices[:n_val] 순서 그대로임
val_labels = torch.tensor([full_train.labels[i] for i in indices[:n_val]])
val_tta_acc = 100. * (val_logits.argmax(dim=1) == val_labels).float().mean().item()
print(f"ConvNeXt-Tiny: val acc (TTA 적용) {val_tta_acc:.2f}%")

# 12번 ensemble용
cnx_val_logits, cnx_test_logits = val_logits, test_logits

Predicting:   0%|          | 0/63 [00:00<?, ?it/s]

Predicting:   0%|          | 0/157 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b49a933c9a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b49a933c9a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

ConvNeXt-Tiny: val acc (TTA 적용) 91.03%


## 9. 모델 2: DINOv2 ViT-g (frozen)

DINOv2
- label 없이 이미지 1억 4천만 장(LVD-142M)으로 학습한 self-supervised ViT
- 같은 이미지에서 자른 다른 crop끼리 비슷한 표현이 나오게 학습해서, 다른 데이터셋에 가져가도 잘 맞는 feature가 나옴
- ViT-g는 parameter 11억 개 정도, token 차원 1536

ViT 구조 (224 입력)
- 이미지를 14x14 patch로 자르면 16x16 = 256개, 각 patch를 vector(token)로 바꿈. 맨 앞에 이미지 전체 요약용 CLS token 하나 붙임
- 블록 40개를 지나면서 self-attention으로 token끼리 정보 주고받음 (멀리 있는 patch끼리도 바로 연결됨)
- timm 기본 입력은 518인데 img_size=224로 만들면 position embedding을 224에 맞게 보간해줌

frozen으로 쓴 이유
- 11억짜리를 fine-tuning하면 T4 메모리, session 시간 안에 절대 못 끝냄
- 학습 안 하면 forward만 하면 되고 feature는 한 번만 뽑으면 됨
- 그 위에 작은 MLP head만 학습 (11번), 이건 몇 분이면 끝남

ConvNeXt 예측은 cnx_*_logits에 이미 저장했으니까 모델이랑 optimizer는 지워서 GPU 메모리 비움

In [10]:
import gc

# 메모리 부족해서 ConvNeXt 관련된 거 지움 (예측값은 cnx_*_logits에 있음)
del model, optimizer, scheduler, scaler
gc.collect()
torch.cuda.empty_cache()

DINO_NAME = "vit_giant_patch14_dinov2.lvd142m"

# num_classes=0: 분류층 없이 feature만 나옴 / img_size=224: position embedding 보간
backbone = timm.create_model(DINO_NAME, pretrained=True, num_classes=0, img_size=IMG_SIZE)
backbone = backbone.to(device).eval().half()  # 학습 안 하니까 float16으로 (메모리, 속도)
for p in backbone.parameters():
    p.requires_grad = False  # frozen

n_params = sum(p.numel() for p in backbone.parameters())
print(f"{DINO_NAME}: parameter 수 {n_params / 1e6:.0f}M, 블록 {len(backbone.blocks)}개, token 차원 {backbone.num_features}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors: downloading bytes:           |  0.00B            

vit_giant_patch14_dinov2.lvd142m: parameter 수 1135M, 블록 40개, token 차원 1536


## 10. DINOv2 feature 추출 (70분 정도)

이미지 한 장의 feature = 1536차원 vector 5개를 이어 붙인 7680차원
- 마지막 4개 블록의 CLS token 4개: 블록마다 요약한 게 조금씩 다름. 뒤쪽일수록 추상적이고 앞쪽은 좀 더 구체적
- 마지막 블록 patch token 256개 평균 1개: 이미지 전체 정보를 골고루 반영
- 중간 블록 출력은 forward hook으로 가져옴. hook은 블록 계산이 끝날 때마다 그 출력을 리스트에 넣어주는 함수. 마지막 블록에 쓰는 LayerNorm(backbone.norm)을 똑같이 걸어서 크기 맞춤
- 여러 블록 CLS token 쓰는 건 DINOv2 논문의 linear 평가 방식 따라한 것. 같은 forward에서 꺼내는 거라 시간은 마지막 블록만 쓸 때랑 똑같음
  - 전에 Linear head로 실험했을 때 마지막 블록만: val 93.67%, 4개 블록: 93.75%
- 추출 끝나면 hook 제거함. 안 지우고 셀 다시 돌리면 hook이 중복으로 붙어서 feature가 꼬임

augmentation 대신 좌우 반전 feature
- feature를 한 번 뽑아서 계속 재사용하는 구조라 랜덤 augmentation은 못 씀 (쓰려면 epoch마다 70분씩 다시 뽑아야 됨)
- 대신 결과가 항상 같은 좌우 반전 feature를 따로 뽑음
  - train: 원본 + 반전 둘 다 학습에 써서 데이터 2배
  - val/test: 두 예측 더함 (TTA)
- DINOv2는 학습할 때 이미 crop, 색 변화 같은 augmentation을 많이 겪어서 이런 변형에 강함

- train 40000장은 augmentation 없는 full_val로 순서대로 뽑은 다음, 4번 인덱스로 train/val 나눔
- 메모리 아끼려고 feature는 float16으로 CPU에 모음 (train 1.2GB, test 0.3GB 정도)

In [11]:
N_BLOCKS = 4  # CLS token 가져올 마지막 블록 수

# 마지막 4개 블록 출력 가로채는 hook
block_outputs = []
hooks = [
    block.register_forward_hook(lambda module, inputs, output: block_outputs.append(output))
    for block in backbone.blocks[-N_BLOCKS:]
]

def embed(images):
    # 마지막 4개 블록 CLS token + 마지막 블록 patch token 평균 -> (batch, 5 x 1536)
    block_outputs.clear()
    tokens = backbone.forward_features(images)  # norm까지 적용됨, (batch, 1 + 256, 1536)
    cls_tokens = [backbone.norm(out)[:, 0] for out in block_outputs]  # hook 출력은 norm 안 돼 있어서 걸어줌
    patch_mean = tokens[:, backbone.num_prefix_tokens:].mean(dim=1)  # CLS 빼고 평균
    return torch.cat(cls_tokens + [patch_mean], dim=1)

@torch.no_grad()
def extract(loader):
    # (원본 feature, 좌우 반전 feature) 반환
    feats, feats_flip = [], []
    for batch in tqdm(loader, desc="Extracting"):
        images = batch[0] if isinstance(batch, (list, tuple)) else batch
        images = upscale(images.to(device, non_blocking=True)).half()
        feats.append(embed(images).cpu())
        feats_flip.append(embed(torch.flip(images, dims=[3])).cpu())
    return torch.cat(feats), torch.cat(feats_flip)

# 순서만 맞으면 돼서 shuffle 없이, batch 128
train_all_loader = DataLoader(full_val, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_feat_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

start = time.time()
train_all_f, train_all_ff = extract(train_all_loader)
test_f, test_ff = extract(test_feat_loader)
print(f"feature 추출 시간: {(time.time() - start) / 60:.1f}분, feature shape: {tuple(train_all_f.shape)}")

# 4번이랑 같은 인덱스로 나눔
train_f, train_ff = train_all_f[indices[n_val:]], train_all_ff[indices[n_val:]]
val_f, val_ff = train_all_f[indices[:n_val]], train_all_ff[indices[:n_val]]
train_labels = torch.tensor([full_train.labels[i] for i in indices[n_val:]])

# hook 안 지우면 다시 돌릴 때 중복으로 붙음. backbone도 이제 필요 없어서 지움
for hook in hooks:
    hook.remove()
del backbone
gc.collect()
torch.cuda.empty_cache()

Extracting:   0%|          | 0/313 [00:00<?, ?it/s]

Extracting:   0%|          | 0/79 [00:00<?, ?it/s]

feature 추출 시간: 65.5분, feature shape: (40000, 7680)


## 11. MLP head 학습

- 입력: 10번에서 뽑은 7680차원 feature, 출력: 100개 클래스 점수. backbone은 그대로 두고 head만 학습
- 구조: Linear(7680 -> 1024), GELU, Dropout 0.5, Linear(1024 -> 100)
  - Linear 하나면 feature 공간을 평면으로만 나눌 수 있는데, hidden layer + GELU 넣으면 휘어진 경계도 됨
  - parameter가 800만 개 정도라 overfitting 걱정돼서 Dropout 0.5, weight decay 0.05 넣음
  - Dropout: 학습 때 hidden 값 절반을 랜덤으로 0으로 만들어서 특정 값에만 의존 못 하게 함. eval() 때는 꺼짐
- 표준화: 차원마다 train 평균 빼고 표준편차로 나눔. 차원마다 범위가 다르면 큰 차원이 학습을 좌우해서. val/test도 train 통계로
- 학습 데이터: train 36000장 원본 + 반전 = 72000개
- 설정: AdamW(lr 1e-3, weight decay 0.05), cosine, label smoothing 0.1, batch 1024, 30 epoch
- 이렇게 정한 이유 (전에 같은 feature, val 4000장으로 실험)
  - Linear head 93.75% -> MLP head 94.30%. seed 3번 바꿔도 94.15~94.30%로 계속 더 높았음
  - hidden 2048, seed 여러 개 평균, test pseudo-labeling도 해봤는데 안 올라서 뺌

In [12]:
def train_head(hidden=1024, weight_decay=0.05, epochs=30, lr=1e-3, batch_size=1024):
    # 표준화한 feature로 MLP head 학습, (head, 표준화 함수) 반환
    X = torch.cat([train_f, train_ff]).to(device).float()  # 원본 + 반전
    y = torch.cat([train_labels, train_labels]).to(device)
    mean, std = X.mean(dim=0), X.std(dim=0) + 1e-6  # val/test에도 train 통계 그대로 씀
    normalize = lambda x: (x.to(device).float() - mean) / std
    X = (X - mean) / std

    torch.manual_seed(2302)
    head = nn.Sequential(
        nn.Linear(X.shape[1], hidden), nn.GELU(), nn.Dropout(0.5), nn.Linear(hidden, 100),
    )
    head = head.to(device).train()
    optimizer = optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    steps = epochs * ((len(X) + batch_size - 1) // batch_size)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)  # batch마다 step
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=device)
        for i in range(0, len(X), batch_size):
            b = perm[i:i + batch_size]
            loss = criterion(head(X[b]), y[b])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
    head.eval()  # Dropout 끔
    return head, normalize

@torch.no_grad()
def predict_head(head, normalize, f, ff):
    # 원본 + 반전 logits 더함 (TTA)
    return (head(normalize(f)) + head(normalize(ff))).cpu()

accuracy = lambda scores: 100. * (scores.argmax(dim=1) == val_labels).float().mean().item()

head, normalize = train_head()
dino_val_logits = predict_head(head, normalize, val_f, val_ff)
dino_test_logits = predict_head(head, normalize, test_f, test_ff)
print(f"DINOv2 + MLP head: val acc (TTA) {accuracy(dino_val_logits):.2f}%")

DINOv2 + MLP head: val acc (TTA) 94.30%


## 12. Ensemble, submission 파일 만들기

- ConvNeXt(CNN, 전체 fine-tuning)랑 DINOv2(ViT, frozen + MLP head)는 구조도 학습 방식도 달라서 틀리는 이미지가 서로 다름. 평균 내면 서로 실수를 메꿔줌
  - 전에 실험: ConvNeXt 91.03%, DINOv2 + MLP 94.30% -> ensemble 94.77%
  - 같은 ConvNeXt 두 개(weight만 다름) 섞었을 땐 +0.02%p밖에 안 올랐음
- 두 모델 logits 크기가 달라서 그냥 더하면 한쪽이 결과를 다 정해버림. softmax로 확률로 바꾸고 1:1로 평균냄
- csv 형식은 샘플이랑 같음. ID(0~9999, test 순서), Prediction(0~99)
- 저장 전에 행 수(10000)랑 열 이름 assert로 확인. submission.csv로 저장되니까 파일 탭에서 다운받아서 제출

In [13]:
import pandas as pd

# 두 모델 softmax 확률 1:1 평균
ens_val_probs = (cnx_val_logits.softmax(dim=1) + dino_val_logits.softmax(dim=1)) / 2
ens_test_probs = (cnx_test_logits.softmax(dim=1) + dino_test_logits.softmax(dim=1)) / 2

print(f"ConvNeXt-Tiny:     val acc {accuracy(cnx_val_logits):.2f}%")
print(f"DINOv2 + MLP head: val acc {accuracy(dino_val_logits):.2f}%")
print(f"ensemble:          val acc {accuracy(ens_val_probs):.2f}%")

submission = pd.DataFrame({
    "ID": np.arange(len(ens_test_probs)),
    "Prediction": ens_test_probs.argmax(dim=1).numpy(),
})
# 제출 형식 확인 (10000행, 첫 열 이름 ID)
assert len(submission) == 10000, len(submission)
assert list(submission.columns) == ["ID", "Prediction"]
submission.to_csv("submission.csv", index=False)
submission.head()

ConvNeXt-Tiny:     val acc 91.03%
DINOv2 + MLP head: val acc 94.30%
ensemble:          val acc 94.77%


,ID,Prediction
0,0,31
1,1,69
2,2,9
3,3,11
4,4,51
